# Project 13: Hotel Booking & Occupancy Analysis

Individual assignment · Dataset: `13_hotel_bookings.csv`

**How to run:** put the CSV in `data/13_hotel_bookings.csv` (or next to this notebook) and use *Run All*. All charts are hand-built SVG, so **no matplotlib** is needed. Tables are saved to `outputs/`.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, SVG, Markdown

# Look for the CSV in the usual places
for p in [Path("data/13_hotel_bookings.csv"), Path("13_hotel_bookings.csv")]:
    if p.exists():
        CSV_PATH = p
        break
else:
    raise FileNotFoundError("Put 13_hotel_bookings.csv in ./data/ or next to this notebook.")

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

# Optional: set to the hotel's room inventory to get a true occupancy %.
# Without it, occupancy is reported as room nights.
TOTAL_ROOMS = None

# Each logical column -> possible names in the CSV (first match wins).
COLUMN_CANDIDATES = {
    "id": ["Booking_ID"],
    "checkin": ["Checkin_Date", "Check_in_Date", "Arrival_Date"],
    "nights": ["Nights", "Stay_Nights", "Num_Nights", "Stay_Duration"],
    "rate": ["Room_Rate", "ADR", "Rate"],
    "room_type": ["Room_Type"],
    "channel": ["Booking_Channel", "Channel"],
    "lead": ["Lead_Time_Days", "Lead_Time"],
    "traveler": ["Traveler_Type", "Travel_Type", "Trip_Type", "Purpose", "Customer_Type"],
    "status": ["Cancellation_Status", "Booking_Status", "Is_Canceled", "Canceled", "Cancelled"],
}


def pick(df, key, required=True):
    for name in COLUMN_CANDIDATES[key]:
        if name in df.columns:
            return name
    if required:
        raise KeyError(f"No column found for '{key}'. Tried {COLUMN_CANDIDATES[key]}. "
                       f"Available: {list(df.columns)}")
    return None


def save(df, name):
    """Save a table to outputs/ and display it in the notebook."""
    df.to_csv(OUT / name)
    print(f"--- {name} ---")
    display(df.round(2))


def is_canceled(series):
    s = series.astype(str).str.strip().str.lower()
    return s.str.contains("cancel") | s.isin(["1", "true", "yes", "y"])

## 1. Load & inspect the data

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head())

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T.head(20))

In [ ]:
C = {k: pick(df, k, required=k in
             ["id", "checkin", "nights", "rate", "room_type", "channel", "lead", "status"])
     for k in COLUMN_CANDIDATES}
C

## 2. Missing values & duplicate booking IDs

In [ ]:
missing = pd.DataFrame({"missing": df.isna().sum(), "pct": (df.isna().mean() * 100).round(2)})
display(missing)
missing.to_csv(OUT / "missing_values.csv")
dup_ids = df[C["id"]].duplicated(keep=False).sum()
print("Rows sharing a duplicated Booking_ID:", dup_ids)
n_before = len(df)
df = df.drop_duplicates(subset=C["id"], keep="first").copy()
print(f"Dropped {n_before - len(df)} duplicate rows (kept first occurrence).")

## 3. Imputation of missing lead time / room rate

Median is used (not mean) because rates and lead times are right-skewed. Group medians (room type for rate, channel for lead time) keep the segment differences that a single global median would erase.

In [ ]:
for key, grp in [("rate", "room_type"), ("lead", "channel")]:
    col = C[key]
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[f"{col}_imputed"] = df[col].isna()
    n = int(df[col].isna().sum())
    df[col] = df[col].fillna(df.groupby(C[grp])[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())  # fallback if a whole group is empty
    print(f"{col}: imputed {n} values with median by {C[grp]}")
df[C["nights"]] = pd.to_numeric(df[C["nights"]], errors="coerce")
bad = df[C["nights"]].isna() | (df[C["nights"]] <= 0)
print(f"Dropping {int(bad.sum())} rows with missing/non-positive nights (cannot compute revenue).")
df = df[~bad].copy()

## 4. Date features

In [ ]:
df[C["checkin"]] = pd.to_datetime(df[C["checkin"]], errors="coerce")
print("Unparseable check-in dates:", int(df[C["checkin"]].isna().sum()))
df = df.dropna(subset=[C["checkin"]]).copy()
df["Arrival_Month"] = df[C["checkin"]].dt.month
df["Arrival_Month_Name"] = df[C["checkin"]].dt.month_name()
df["Arrival_Weekday"] = df[C["checkin"]].dt.day_name()
df["Year_Month"] = df[C["checkin"]].dt.to_period("M").astype(str)

df["Canceled"] = is_canceled(df[C["status"]])
nights, rate = C["nights"], C["rate"]

## 5. KPIs

In [ ]:
active = df[~df["Canceled"]]
kpi = pd.Series({
    "Total bookings": len(df),
    "Total room nights (all bookings)": df[nights].sum(),
    "Room nights (non-canceled)": active[nights].sum(),
    "Canceled bookings": int(df["Canceled"].sum()),
    "Cancellation %": df["Canceled"].mean() * 100,
}, name="value")
save(kpi.to_frame(), "kpi_summary.csv")

## 6. Revenue (non-canceled bookings)

In [ ]:
df["Booking_Revenue"] = np.where(df["Canceled"], 0.0, df[nights] * df[rate])
active = df[~df["Canceled"]]
print("Realized revenue (non-canceled):", round(active["Booking_Revenue"].sum(), 2))

## 7. Lead-time bands

In [ ]:
bins = [-np.inf, 7, 30, 90, np.inf]
labels = ["Last-Minute (0-7d)", "Short-Term (8-30d)", "Medium (31-90d)", "Advance (90d+)"]
df["Lead_Band"] = pd.cut(df[C["lead"]], bins=bins, labels=labels)
active = df[~df["Canceled"]]
save(df["Lead_Band"].value_counts().reindex(labels).to_frame("bookings"), "lead_time_bands.csv")

## 8. Room types

In [ ]:
room = active.groupby(C["room_type"]).agg(
    bookings=(C["id"], "count"),
    room_nights=(nights, "sum"),
    avg_room_rate=(rate, "mean"),
    revenue=("Booking_Revenue", "sum"),
)
room["revenue_share_%"] = room["revenue"] / room["revenue"].sum() * 100
room["room_night_share_%"] = room["room_nights"] / room["room_nights"].sum() * 100
room = room.sort_values("revenue", ascending=False)
save(room, "room_type_summary.csv")

In [ ]:
rc = (active.groupby([C["room_type"], C["channel"]])
      .agg(bookings=(C["id"], "count"), room_nights=(nights, "sum"),
           avg_rate=(rate, "mean"), revenue=("Booking_Revenue", "sum")))
save(rc, "room_type_by_channel.csv")

## 9. Booking channels

In [ ]:
chan = df.groupby(C["channel"]).agg(total_bookings=(C["id"], "count"))
chan["active_bookings"] = active.groupby(C["channel"])[C["id"]].count()
chan["revenue"] = active.groupby(C["channel"])["Booking_Revenue"].sum()
chan["volume_rank"] = chan["total_bookings"].rank(ascending=False, method="min").astype(int)
chan["revenue_rank"] = chan["revenue"].rank(ascending=False, method="min").astype(int)
chan["revenue_per_booking"] = chan["revenue"] / chan["active_bookings"]
chan = chan.sort_values("revenue", ascending=False)
save(chan, "channel_ranking.csv")

## 10. Cancellations

In [ ]:
canc_ch = df.groupby(C["channel"]).agg(bookings=(C["id"], "count"),
                                       canceled=("Canceled", "sum"))
canc_ch["cancel_rate_%"] = canc_ch["canceled"] / canc_ch["bookings"] * 100
canc_ch = canc_ch.sort_values("cancel_rate_%", ascending=False)
save(canc_ch, "cancellation_by_channel.csv")

In [ ]:
canc_lead = df.groupby("Lead_Band", observed=False).agg(bookings=(C["id"], "count"),
                                                        canceled=("Canceled", "sum"))
canc_lead["cancel_rate_%"] = canc_lead["canceled"] / canc_lead["bookings"] * 100
save(canc_lead, "cancellation_by_lead_band.csv")

In [ ]:
canc_x = df.pivot_table(index=C["channel"], columns="Lead_Band", values="Canceled",
                        aggfunc="mean", observed=False) * 100
save(canc_x, "cancellation_channel_x_lead_band.csv")

## 11. Peak months & occupancy trend

In [ ]:
monthly = active.groupby("Year_Month").agg(bookings=(C["id"], "count"),
                                           room_nights=(nights, "sum"),
                                           revenue=("Booking_Revenue", "sum"))
if TOTAL_ROOMS:
    days = pd.PeriodIndex(monthly.index, freq="M").days_in_month
    monthly["occupancy_%"] = monthly["room_nights"] / (TOTAL_ROOMS * days) * 100
save(monthly, "monthly_trend.csv")
top3 = monthly.sort_values("revenue", ascending=False).head(3)
save(top3, "top3_revenue_months.csv")

In [ ]:
by_moy = (active.groupby("Arrival_Month_Name")
          .agg(room_nights=(nights, "sum"), revenue=("Booking_Revenue", "sum"))
          .sort_values("revenue", ascending=False))
save(by_moy, "revenue_by_calendar_month.csv")
wd = active.groupby("Arrival_Weekday").agg(arrivals=(C["id"], "count"))
wd = wd.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
save(wd, "arrivals_by_weekday.csv")  # supports front-desk staffing

## 12. Traveler type

In [ ]:
if C["traveler"]:
    trav = active.groupby(C["traveler"]).agg(bookings=(C["id"], "count"),
                                             avg_nights=(nights, "mean"),
                                             median_nights=(nights, "median"),
                                             avg_rate=(rate, "mean"))
    save(trav, "stay_by_traveler_type.csv")
else:
    trav = None
    print("\nNo traveler-type column found; skipping Task 11 (edit COLUMN_CANDIDATES).")

## 13. NumPy rate statistics

In [ ]:
r = active[rate].to_numpy(dtype=float)
rate_stats = pd.Series({"mean": np.mean(r), "median": np.median(r), "std_dev": np.std(r, ddof=1)},
                       name="room_rate")
save(rate_stats.to_frame(), "room_rate_numpy_stats.csv")

## 14. IQR outliers (high-value bookings)

In [ ]:
rev = active["Booking_Revenue"].to_numpy(dtype=float)
q1, q3 = np.percentile(rev, [25, 75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
high = active[active["Booking_Revenue"] > upper]
print(f"IQR: Q1={q1:.0f} Q3={q3:.0f} IQR={iqr:.0f} upper fence={upper:.0f} "
      f"-> {len(high)} high-value bookings ({len(high)/len(active)*100:.1f}%)")
high.sort_values("Booking_Revenue", ascending=False).to_csv(OUT / "high_value_outliers.csv", index=False)
display(high.groupby(C["room_type"]).size().to_frame("high_value_bookings"))

## 15. Charts (hand-built SVG, no plotting library)

In [ ]:
def svg_bars(series, title, color="#2a6fdb", w=640, h=260):
    vals = [float(v) for v in series.values]; names = [str(i) for i in series.index]
    top = max(vals) if vals and max(vals) > 0 else 1
    pad_l, pad_b, pad_t = 60, 60, 30
    bw = (w - pad_l - 10) / max(len(vals), 1)
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" font-family="sans-serif" font-size="11">',
           f'<text x="{w/2}" y="18" text-anchor="middle" font-size="14" font-weight="bold">{title}</text>',
           f'<line x1="{pad_l}" y1="{h-pad_b}" x2="{w-10}" y2="{h-pad_b}" stroke="#888"/>',
           f'<text x="{pad_l-6}" y="{pad_t+4}" text-anchor="end">{top:,.0f}</text>',
           f'<text x="{pad_l-6}" y="{h-pad_b}" text-anchor="end">0</text>']
    for i, (n, v) in enumerate(zip(names, vals)):
        bh = (h - pad_b - pad_t) * v / top
        x = pad_l + i * bw
        out.append(f'<rect x="{x+3:.1f}" y="{h-pad_b-bh:.1f}" width="{max(bw-6,1):.1f}" height="{bh:.1f}" fill="{color}"/>')
        out.append(f'<text x="{x+bw/2:.1f}" y="{h-pad_b+12}" text-anchor="end" '
                   f'transform="rotate(-40 {x+bw/2:.1f} {h-pad_b+12})">{n}</text>')
    return "".join(out) + "</svg>"


def svg_line(series, title, color="#1b9e5a", w=640, h=260):
    vals = [float(v) for v in series.values]; names = [str(i) for i in series.index]
    top = max(vals) if vals and max(vals) > 0 else 1
    pad_l, pad_b, pad_t = 60, 60, 30
    step = (w - pad_l - 20) / max(len(vals) - 1, 1)
    pts = [(pad_l + i * step, h - pad_b - (h - pad_b - pad_t) * v / top) for i, v in enumerate(vals)]
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" font-family="sans-serif" font-size="11">',
           f'<text x="{w/2}" y="18" text-anchor="middle" font-size="14" font-weight="bold">{title}</text>',
           f'<line x1="{pad_l}" y1="{h-pad_b}" x2="{w-10}" y2="{h-pad_b}" stroke="#888"/>',
           f'<text x="{pad_l-6}" y="{pad_t+4}" text-anchor="end">{top:,.0f}</text>',
           f'<text x="{pad_l-6}" y="{h-pad_b}" text-anchor="end">0</text>',
           '<polyline fill="none" stroke="%s" stroke-width="2" points="%s"/>' %
           (color, " ".join(f"{x:.1f},{y:.1f}" for x, y in pts))]
    for (x, y), n in zip(pts, names):
        out.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3" fill="{color}"/>')
        out.append(f'<text x="{x:.1f}" y="{h-pad_b+12}" text-anchor="end" '
                   f'transform="rotate(-40 {x:.1f} {h-pad_b+12})">{n}</text>')
    return "".join(out) + "</svg>"

In [ ]:
occ_col = "occupancy_%" if TOTAL_ROOMS else "room_nights"
occ_title = "Monthly occupancy (%)" if TOTAL_ROOMS else "Monthly occupancy (room nights)"
band_counts = canc_lead["cancel_rate_%"].fillna(0)
charts = [
    ("Monthly realized revenue", svg_line(monthly["revenue"], "Monthly realized revenue")),
    ("Occupancy", svg_line(monthly[occ_col], occ_title)),
    ("Revenue by room type", svg_bars(room["revenue"], "Revenue by room type")),
    ("Cancellation by channel", svg_bars(canc_ch["cancel_rate_%"], "Cancellation % by booking channel", "#d64545")),
    ("Cancellation by lead band", svg_bars(band_counts, "Cancellation % by lead-time band", "#d68a2a")),
    ("Arrivals by weekday", svg_bars(wd["arrivals"], "Arrivals by weekday (front-desk load)", "#7a4fd1")),
]
for _, svg in charts:
    display(SVG(data=svg))

# Also keep the standalone HTML version
html = ("<!doctype html><html><head><meta charset='utf-8'><title>Hotel charts</title></head>"
        "<body style='font-family:sans-serif'><h1>Hotel Booking Charts</h1>" +
        "".join(f"<div style='margin:16px 0'>{s}</div>" for _, s in charts) + "</body></html>")
(OUT / "charts.html").write_text(html, encoding="utf-8")

## 16. Data-driven recommendations

In [ ]:
top_room = room.index[0]
low_rate_room = room["avg_room_rate"].idxmin()
best_ch = chan.index[0]
worst_cancel_ch = canc_ch.index[0]
worst_band = canc_lead["cancel_rate_%"].idxmax()
peak = ", ".join(top3.index)
busy_days = wd["arrivals"].nlargest(2).index.tolist()
slow_days = wd["arrivals"].nsmallest(1).index.tolist()

recs = [
    f"**Dynamic pricing in peak months.** {peak} are the top-3 revenue months. Raise rates and add "
    f"minimum-stay rules there; discount shoulder months to protect occupancy.",
    f"**Lean on the highest-earning room type.** {top_room} generates the largest revenue share "
    f"({room.loc[top_room, 'revenue_share_%']:.1f}%). Prioritise its availability on high-demand dates "
    f"and use upgrade offers from {low_rate_room} (lowest avg rate) to lift average revenue per night.",
    f"**Rebalance the channel mix.** {best_ch} ranks first on revenue. Check commission cost per channel, "
    f"push direct-booking incentives (member rates, free breakfast) to cut commissions, and review "
    f"channels where volume rank is better than revenue rank.",
    f"**Tighten cancellation policy where risk is highest.** {worst_cancel_ch} has the highest cancellation "
    f"rate ({canc_ch.loc[worst_cancel_ch, 'cancel_rate_%']:.1f}%) and the {worst_band} band cancels most "
    f"({canc_lead.loc[worst_band, 'cancel_rate_%']:.1f}%). Use deposits or non-refundable rates there, and "
    f"consider controlled overbooking sized to these rates.",
    f"**Staff the front desk to arrival patterns.** Arrivals peak on {busy_days[0]} and {busy_days[1]} "
    f"and are lightest on {slow_days[0]}. Schedule extra check-in staff on peak days and use quiet days "
    f"for training or leave. Offer a corporate/VIP fast check-in for the {len(high)} high-value bookings "
    f"above the IQR fence.",
]

In [ ]:
lines = ["# Hotel Booking & Occupancy Analysis: Report", "",
         "## Key figures",
         f"- Total bookings: {len(df):,}",
         f"- Total room nights: {df[nights].sum():,.0f} ({active[nights].sum():,.0f} non-canceled)",
         f"- Cancellation rate: {df['Canceled'].mean()*100:.1f}%",
         f"- Realized revenue: {active['Booking_Revenue'].sum():,.0f}",
         f"- Daily rate: mean {rate_stats['mean']:.2f}, median {rate_stats['median']:.2f}, "
         f"std {rate_stats['std_dev']:.2f}",
         f"- IQR upper fence for booking value: {upper:,.0f} ({len(high)} outliers)",
         f"- Top 3 revenue months: {peak}", "",
         "## Five recommendations"] + [f"{i}. {t}" for i, t in enumerate(recs, 1)]
(OUT / "report.md").write_text("\n".join(lines), encoding="utf-8")
display(Markdown("\n".join(lines)))
print(f"Done. Files saved in {OUT.resolve()}")